# 05 · Agro, religião e transferências (hipóteses A2, A3 e A4)

**Testes confirmatórios.** Especificações registradas em `docs/hipoteses.md` (seção
"A2, A3 e A4") antes de qualquer regressão com as medidas novas.

**De onde vêm as hipóteses.** No modelo da A1 (`02_tamanho`), três controles chamaram a
atenção: o peso do agro, a % de evangélicos e a renda vinda de transferências. Testar essas
pistas com as mesmas variáveis seria circular. Por isso cada hipótese usa uma **medida
nova**, e a variável que sugeriu a pista continua no modelo como controle:

| hipótese | medida nova | fonte | controle que sugeriu a pista |
|---|---|---|---|
| A2 · agro | expansão da soja 2021→2025, em % do território | IBGE/PAM | `pct_va_agro_2021` |
| A3 · religião | crescimento evangélico 2010→2022, em pp | Censos 2010 e 2022 | `pct_evangelica` |
| A4 · transferências | famílias no programa por 100 habitantes, 2022 e 2026 | MDS | `pct_renda_outras_fontes` |

**Base comum:** o modelo principal da A1 (M3): variação do PT em logit entre os 1ºs turnos
de 2022 e 2026, por AMC, comparando só AMCs da mesma região imediata, com todos os controles.

In [ ]:
import warnings

import altair as alt
import polars as pl
import pyfixest as pf

from kingsrace import paths
from kingsrace.features.base import base_variacao
from kingsrace.viz import tema  # noqa: F401  (registra e ativa o tema do projeto)
from kingsrace.viz.tema import DIVERGENTE, SERIES, TINTA

alt.renderers.enable("jupyter", offline=True)
warnings.filterwarnings("ignore", message=".*singleton fixed effect")

P = paths.PROCESSED
base = base_variacao(
    pl.read_parquet(P / "painel_amc.parquet"),
    pl.read_parquet(P / "covariaveis_amc.parquet"),
    pl.read_parquet(P / "municipios.parquet"),
    de=(2022, 1),
    para=(2026, 1),
).with_columns(log_renda=pl.col("renda_pc_mediana").log())

CONTROLES = [
    "log_aptos_de",
    "pt_2p_logit_de",
    "pct_urbana",
    "idade_mediana",
    "pct_evangelica",
    "log_renda",
    "taxa_alfabetizacao",
    "pct_preta_parda",
    "pct_va_agro_2021",
    "pct_renda_outras_fontes",
]
cluster = {"CRV1": "cd_regiao_imediata"}

## A2, A3 e A4a · A medida nova explica a queda além dos controles?

Cada modelo acrescenta a medida nova ao M3 da A1. Hipóteses registradas:

- **A2:** onde a soja mais avançou, o PT perdeu **mais** (coeficiente negativo);
- **A3:** onde os evangélicos mais cresceram, o PT perdeu **mais** (negativo);
- **A4a:** onde o Bolsa Família é maior em 2026, o PT perdeu **menos** (positivo).

In [ ]:
testes = {
    "A2 · expansão da soja": (
        ["exp_soja_pct_territorio", "soja_2021_pct_territorio"],
        "exp_soja_pct_territorio",
    ),
    "A3 · crescimento evangélico": (["delta_evangelica"], "delta_evangelica"),
    "A4a · Bolsa Família 2026": (["familias_bf_100hab_2026"], "familias_bf_100hab_2026"),
}
linhas = []
for nome, (extras, alvo) in testes.items():
    dados = base.drop_nulls(["delta_logit", *CONTROLES, *extras]).to_pandas()
    fit = pf.feols(
        f"delta_logit ~ {' + '.join([*extras, *CONTROLES])} | cd_regiao_imediata",
        dados,
        vcov=cluster,
    )
    t = fit.tidy().loc[alvo]
    dp = dados[alvo].std()
    linhas.append(
        {
            "teste": nome,
            "coeficiente": t["Estimate"],
            "ic_inf": t["2.5%"],
            "ic_sup": t["97.5%"],
            "p": t["Pr(>|t|)"],
            "dp_variavel": dp,
            "efeito_1dp": t["Estimate"] * dp,
            "ic_inf_1dp": t["2.5%"] * dp,
            "ic_sup_1dp": t["97.5%"] * dp,
            "amcs": fit._N,
        }
    )
resultados = pl.DataFrame(linhas)
resultados.select("teste", "coeficiente", "ic_inf", "ic_sup", "p", "amcs")

**Como ler:** efeito de 1 desvio-padrão de cada medida nova sobre a variação do PT (logit),
com IC de 95%. À direita de zero, a medida está associada a uma queda **menor** do PT; à
esquerda, a uma queda **maior**. Para comparação, o efeito do tamanho do município (A1) é de
cerca de +0,027 por desvio-padrão.

In [ ]:
base_r = alt.Chart(resultados).encode(
    y=alt.Y("teste:N", title=None, sort=list(testes)),
    tooltip=[
        alt.Tooltip("teste:N", title="Teste"),
        alt.Tooltip("efeito_1dp:Q", title="Efeito de 1 dp (Δ logit)", format="+.4f"),
        alt.Tooltip("ic_inf_1dp:Q", title="IC 95% inf.", format="+.4f"),
        alt.Tooltip("ic_sup_1dp:Q", title="IC 95% sup.", format="+.4f"),
        alt.Tooltip("p:Q", title="p", format=".3g"),
        alt.Tooltip("amcs:Q", title="AMCs"),
    ],
)
zero = alt.Chart().mark_rule(color=TINTA["apagada"], strokeDash=[4, 4]).encode(x=alt.datum(0))
(
    zero
    + base_r.mark_rule(strokeWidth=2, color=SERIES[0]).encode(
        x=alt.X("ic_inf_1dp:Q", title="Efeito de 1 desvio-padrão sobre Δ logit do PT"),
        x2="ic_sup_1dp:Q",
    )
    + base_r.mark_circle(size=90, opacity=1, color=SERIES[0]).encode(x="efeito_1dp:Q")
).properties(
    width=520, height=150, title="Soja e evangélicos: nada além dos controles; Bolsa Família: sim"
)

**Resultado:**

- **A2 não se sustenta.** A expansão da soja não acrescenta nada ao que o peso do agro
  (já no modelo) explica.
- **A3 não se sustenta.** O crescimento evangélico entre os Censos não prevê a queda,
  dado o nível de 2022.
- **A4a se sustenta.** Onde o Bolsa Família alcança mais famílias, o PT perdeu menos,
  mesmo controlando a renda, a parcela da renda vinda de transferências e o resto.

## A4 · Quando o programa virou "do Lula"

A formulação original: comparar 2022 (Auxílio Brasil, governo Bolsonaro) com 2026 (Bolsa
Família, governo Lula). Num painel com os dois anos, o desfecho é o **nível** do PT (logit
da fatia entre os dois maiores) e a variável é a cobertura do programa em cada ano. A
interação cobertura × 2026 mede quanto a associação mudou. Todos os controles e os efeitos
fixos de região imediata podem ter efeitos diferentes em cada ano.

In [ ]:
ctrl_nivel = [c for c in CONTROLES if c != "pt_2p_logit_de"]
cob22, cob26 = "familias_ab_100hab_2022", "familias_bf_100hab_2026"
completos = base.drop_nulls(["pt_2p_logit_de", "pt_2p_logit_para", cob22, cob26, *ctrl_nivel])
anos = [
    completos.select(
        "cd_regiao_imediata", *ctrl_nivel, lg="pt_2p_logit_de", cobertura=cob22
    ).with_columns(d26=pl.lit(0)),
    completos.select(
        "cd_regiao_imediata", *ctrl_nivel, lg="pt_2p_logit_para", cobertura=cob26
    ).with_columns(d26=pl.lit(1)),
]
painel = pl.concat(anos).with_columns(
    *[(pl.col(c) * pl.col("d26")).alias(f"{c}_x26") for c in ["cobertura", *ctrl_nivel]],
    ri_ano=pl.col("cd_regiao_imediata").cast(pl.String) + "_" + pl.col("d26").cast(pl.String),
)
termos = ["cobertura", "cobertura_x26", *ctrl_nivel, *(f"{c}_x26" for c in ctrl_nivel)]
a4 = pf.feols(f"lg ~ {' + '.join(termos)} | ri_ano", painel.to_pandas(), vcov=cluster)
a4.tidy().loc[["cobertura", "cobertura_x26"], ["Estimate", "2.5%", "97.5%", "Pr(>|t|)"]]

**Como ler:** a associação entre cobertura do programa e voto no PT (logit, por família
beneficiária a cada 100 habitantes), em cada ano, com IC de 95%. O valor de 2026 é a soma
do coeficiente base com a interação.

In [ ]:
vcov = a4._vcov
nomes_coef = list(a4.coef().index)
i, j = nomes_coef.index("cobertura"), nomes_coef.index("cobertura_x26")
b22 = a4.coef()["cobertura"]
b26 = b22 + a4.coef()["cobertura_x26"]
ep22 = vcov[i, i] ** 0.5
ep26 = (vcov[i, i] + vcov[j, j] + 2 * vcov[i, j]) ** 0.5
por_ano = pl.DataFrame(
    {
        "ano": ["2022 · Auxílio Brasil (Bolsonaro)", "2026 · Bolsa Família (Lula)"],
        "coef": [b22, b26],
        "ic_inf": [b22 - 1.96 * ep22, b26 - 1.96 * ep26],
        "ic_sup": [b22 + 1.96 * ep22, b26 + 1.96 * ep26],
    }
)
base_a = alt.Chart(por_ano).encode(
    y=alt.Y("ano:N", title=None),
    tooltip=[
        alt.Tooltip("ano:N", title="Ano"),
        alt.Tooltip("coef:Q", title="Coeficiente", format="+.4f"),
        alt.Tooltip("ic_inf:Q", title="IC 95% inf.", format="+.4f"),
        alt.Tooltip("ic_sup:Q", title="IC 95% sup.", format="+.4f"),
    ],
)
(
    zero
    + base_a.mark_rule(strokeWidth=2, color=DIVERGENTE["pt"]).encode(
        x=alt.X("ic_inf:Q", title="Associação entre cobertura do programa e voto no PT (logit)"),
        x2="ic_sup:Q",
    )
    + base_a.mark_circle(size=90, opacity=1, color=DIVERGENTE["pt"]).encode(x="coef:Q")
).properties(
    width=520,
    height=110,
    title="A associação com o programa dobrou quando ele passou a ser do governo Lula",
)

**A4 se sustenta.** Em 2022, já havia associação positiva entre a cobertura do Auxílio
Brasil e o voto no PT (com tudo controlado). Em 2026, com o Bolsa Família no governo Lula,
ela **praticamente dobrou**.

**Cuidados:**

- **Ecológico:** municípios com mais beneficiários votaram mais no PT; isso não mostra que
  os beneficiários mudaram de voto.
- **Os programas não são iguais:** o Auxílio Brasil de set/2022 pagava R$ 600 e tinha regras
  diferentes; a cobertura de 2026 é um pouco menor (11,9 contra 13,2 famílias por 100
  habitantes em média). O painel compara a **associação**, não o tamanho do programa.
- **Causalidade:** a cobertura acompanha a pobreza, que os controles tratam só em parte
  (renda mediana e parcela de transferências).

## Veredito

| hipótese | resultado |
|---|---|
| A2 · expansão da soja | não se sustenta |
| A3 · crescimento evangélico | não se sustenta |
| A4 · associação com o programa em 2026 vs 2022 | **sustenta-se**: praticamente dobrou |
| A4a · Bolsa Família e queda menor | **sustenta-se** |

As pistas de agro e religião vistas no M3 da A1 **não se confirmam** com medidas novas. A
das transferências **se confirma** com uma medida independente (dados do MDS, não do Censo).